# 06 - Integracion con Streamlit

Este notebook explica como la aplicacion usa el modelo entrenado. La app permite tres entradas: imagen subida, foto tomada con camara y camara en vivo.

La regla importante es que la app debe usar el mismo preprocesamiento que el entrenamiento para que las predicciones sean consistentes.


## Bloque 1: preparar rutas e importar funciones de la app

Este bloque carga funciones de `app_streamlit/utils.py`. Esas funciones se encargan de leer imagenes, aplicar preprocesamiento, cargar pesos CUDA y convertir la salida sigmoid en una clase.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app_streamlit.utils import CUDA_WEIGHTS_PATH, check_runtime_dependencies, classify_probability

print("Ruta de pesos CUDA:", CUDA_WEIGHTS_PATH.relative_to(ROOT))
print("Dependencias faltantes:", check_runtime_dependencies())


Ruta de pesos CUDA: models/cuda_weights.bin
Dependencias faltantes: []


## Bloque 2: comprobar si el modelo esta disponible

La app necesita `models/cuda_weights.bin`. Este archivo se genera cuando entrenas manualmente con `python -m src.train`. Si no existe, Streamlit puede abrir, pero no podra hacer predicciones.


In [2]:
if CUDA_WEIGHTS_PATH.exists():
    print("Modelo disponible para la aplicacion.")
    print("Tamano en KB:", round(CUDA_WEIGHTS_PATH.stat().st_size / 1024, 2))
else:
    print("No existe models/cuda_weights.bin. Ejecuta manualmente: python -m src.train")


Modelo disponible para la aplicacion.
Tamano en KB: 1032.52


## Bloque 3: explicar la salida sigmoid

El MLP tiene una salida sigmoid. Ese numero se interpreta como probabilidad de `yawn`. Si la probabilidad es mayor o igual a 0.5, la app muestra `Bostezo detectado`; si es menor, muestra `No se detecta bostezo`.


In [3]:
for probability in [0.10, 0.49, 0.50, 0.85]:
    label = classify_probability(probability)
    print(f"Probabilidad yawn={probability:.2f} -> {label}")


Probabilidad yawn=0.10 -> No se detecta bostezo
Probabilidad yawn=0.49 -> No se detecta bostezo
Probabilidad yawn=0.50 -> Bostezo detectado
Probabilidad yawn=0.85 -> Bostezo detectado


## Bloque 4: revisar componentes principales de Streamlit

Este bloque confirma que existe el archivo de la aplicacion. La interfaz usa una distribucion en columnas: a la izquierda aparece la imagen o camara, y a la derecha el resultado con probabilidades de `yawn` y `no_yawn`.


In [4]:
app_path = ROOT / "app_streamlit" / "streamlit_app.py"
utils_path = ROOT / "app_streamlit" / "utils.py"
print("streamlit_app.py:", app_path.exists())
print("utils.py:", utils_path.exists())


streamlit_app.py: True
utils.py: True


## Bloque 5: revisar si la app tiene camara en vivo

La camara en vivo usa `streamlit-webrtc` y `av`. En despliegue web puede requerir configuracion STUN/TURN para que el navegador y el servidor puedan intercambiar video.


In [5]:
text = (ROOT / "app_streamlit" / "streamlit_app.py").read_text(encoding="utf-8")
keywords = ["webrtc_streamer", "RTC_CONFIGURATION", "LiveYawnProcessor", "prediction_panel"]
for keyword in keywords:
    print(keyword, "->", keyword in text)


webrtc_streamer -> True
RTC_CONFIGURATION -> True
LiveYawnProcessor -> True
prediction_panel -> True


## Bloque 6: comando manual para abrir la app

Este comando inicia Streamlit desde WSL. Queda comentado para que el notebook no abra servidores automaticamente. Cuando lo ejecutes, abre la URL que muestre Streamlit.


In [6]:
# !python -m streamlit run app_streamlit/streamlit_app.py


## Bloque 7: que debe mostrarse en la aplicacion

La app debe mostrar la prediccion de manera consistente en los tres modos. El resultado esperado es ver la clase final y dos barras de probabilidad: probabilidad de bostezo y probabilidad de no bostezo.


In [7]:
print("Checklist visual de Streamlit:")
print("- Imagen subida: imagen a la izquierda, resultado a la derecha.")
print("- Foto tomada: foto del control de camara sin duplicarse, resultado a la derecha.")
print("- Camara en vivo: video a la izquierda, probabilidades actualizandose a la derecha.")
print("- No debe aparecer el texto 'Confianza de la clase'.")


Checklist visual de Streamlit:
- Imagen subida: imagen a la izquierda, resultado a la derecha.
- Foto tomada: foto del control de camara sin duplicarse, resultado a la derecha.
- Camara en vivo: video a la izquierda, probabilidades actualizandose a la derecha.
- No debe aparecer el texto 'Confianza de la clase'.
